# Recover the existing full-M1 evidence ZIP — no training

Use this notebook after notebook12 completed but its Kaggle ZIP was not downloaded. Saved notebook12 outputs report successful final private HF upload and hash round-trip at revision `a98bc31bcb3f7c404ea91ead35689301ab0c40d3`, path `shared/m1-full-qwen17-v1/m1-full-qwen17-v1-hpcajqfx`.

Connect to a fresh Kaggle kernel with the same HF_TOKEN secret enabled, then run the single cell below. It is independent of the old session's variables/files and requires no GPU, Git clone, model weights or retraining. It downloads only root text evidence/checkpoint metadata, retrieves the separately uploaded final receipt, verifies hashes and rebuilds a ZIP. Model/optimizer binaries remain private on HF. The rebuilt ZIP's bytes/timestamps may differ from the lost ZIP; the pinned file contents/provenance are preserved and a recovery receipt is added.

After the cell finishes, download the printed ZIP path from Kaggle Output and preserve it for independent result review. Recovery does not change scores/gates or promote canonical M1. A successful training execution is distinct from passing all learning gates.


In [1]:
from pathlib import Path, PurePosixPath
import hashlib, json, shutil, tempfile, zipfile

restore_working_root = Path('/kaggle/working')
assert restore_working_root.is_dir(), 'Select a Kaggle kernel; no GPU is required.'
try:
    from huggingface_hub import HfApi, hf_hub_download
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'huggingface-hub==1.33.0'], check=True)
    from huggingface_hub import HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient

restore_token = UserSecretsClient().get_secret('HF_TOKEN')
restore_repo = 'tharundevulapalli/persistent-llm-unlearning-checkpoints'
restore_prefix = 'shared/m1-full-qwen17-v1/m1-full-qwen17-v1-hpcajqfx'
restore_revision = 'a98bc31bcb3f7c404ea91ead35689301ab0c40d3'
restore_source_commit = '35b961e914a57a0fc09cd4c9d02e634670778d73'
restore_adapter_sha = '674aa28ee0bfd806829447844e8eb50c23f768ff1233cfc702b82a5542d12a99'
restore_api = HfApi(token=restore_token)
restore_info = restore_api.repo_info(restore_repo, repo_type='model')
assert restore_info.private, 'Expected the existing private artifact repository.'
restore_cache = Path(tempfile.mkdtemp(prefix='full-m1-recovery-cache-', dir=restore_working_root))
restore_dir = Path(tempfile.mkdtemp(prefix='m1-full-hpcajqfx-restored-', dir=restore_working_root))

# The receipt was uploaded in a later commit than the artifact folder itself.
restore_receipt_revision = restore_info.sha
assert isinstance(restore_receipt_revision, str) and len(restore_receipt_revision) == 40, 'Cannot pin the receipt commit.'
restore_receipt_file = hf_hub_download(repo_id=restore_repo, repo_type='model', filename=restore_prefix + '/persistence.json', revision=restore_receipt_revision, token=restore_token, local_dir=restore_cache)
restore_receipt_bytes = Path(restore_receipt_file).read_bytes()
restore_receipt = json.loads(restore_receipt_bytes)
assert restore_receipt['repo_id'] == restore_repo and restore_receipt['path'] == restore_prefix
assert restore_receipt['artifact_revision'] == restore_revision and restore_receipt['all_artifact_files_verified']
assert restore_receipt['artifact_files_sha256']['adapter/adapter_model.safetensors'] == restore_adapter_sha

restore_files = []
for restore_name in restore_api.list_repo_files(restore_repo, repo_type='model', revision=restore_revision):
    if not restore_name.startswith(restore_prefix + '/'):
        continue
    restore_relative = restore_name[len(restore_prefix) + 1:]
    restore_parts = PurePosixPath(restore_relative).parts
    assert all(part not in ('.', '..') for part in restore_parts)
    restore_root_text = len(restore_parts) == 1 and PurePosixPath(restore_relative).suffix in {'.json', '.jsonl', '.txt'}
    restore_checkpoint_meta = (len(restore_parts) == 3 and restore_parts[0] == 'checkpoints'
        and restore_parts[1] in {'attempt-000300', 'attempt-000600', 'attempt-000900', 'attempt-001200'}
        and restore_parts[2] in {'checkpoint.json', 'artifact_manifest.json', 'persistence.json'})
    if restore_root_text or restore_checkpoint_meta:
        restore_files.append((restore_name, restore_relative))
restore_required = {'summary.json', 'plan.json', 'artifact_manifest.json', 'training.jsonl', 'checkpoint_receipts.jsonl'}
assert restore_required <= {relative for _, relative in restore_files}, 'Endpoint snapshot is missing required evidence; do not retrain.'
for restore_index, (restore_name, restore_relative) in enumerate(sorted(restore_files), 1):
    restore_download = hf_hub_download(repo_id=restore_repo, repo_type='model', filename=restore_name, revision=restore_revision, token=restore_token, local_dir=restore_cache)
    restore_target = restore_dir / restore_relative
    restore_target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(restore_download, restore_target)
    print(f'Recovered {restore_index}/{len(restore_files)}: {restore_relative}', flush=True)
# Preserve the final receipt separately fetched from its immutable later commit.
(restore_dir / 'persistence.json').write_bytes(restore_receipt_bytes)
restore_summary = json.loads((restore_dir / 'summary.json').read_text())
restore_manifest = json.loads((restore_dir / 'artifact_manifest.json').read_text())
assert restore_summary['execution_complete'] and restore_summary['optimizer_updates'] == 1200
assert restore_summary['plan']['source_git_commit'] == restore_source_commit
assert restore_summary['artifact_files_sha256'] == restore_receipt['artifact_files_sha256'] == restore_manifest['artifact_files_sha256']
for restore_name, restore_digest in restore_summary['prediction_files_sha256'].items():
    assert PurePosixPath(restore_name).name == restore_name
    assert hashlib.sha256((restore_dir / restore_name).read_bytes()).hexdigest() == restore_digest, restore_name
restore_steps = [json.loads(line) for line in (restore_dir / 'training.jsonl').read_text().splitlines()]
assert len(restore_steps) == 1200 and [row['attempt'] for row in restore_steps] == list(range(1, 1201))
assert sum(row['update_applied'] for row in restore_steps) == restore_summary['optimizer_updates']
restore_checkpoints = [json.loads(line) for line in (restore_dir / 'checkpoint_receipts.jsonl').read_text().splitlines()]
assert [row['attempt'] for row in restore_checkpoints] == [300, 600, 900, 1200]
assert all(row['all_artifact_files_verified'] for row in restore_checkpoints)
restore_hashes = {str(file.relative_to(restore_dir)): hashlib.sha256(file.read_bytes()).hexdigest() for file in sorted(restore_dir.rglob('*')) if file.is_file()}
(restore_dir / 'recovery.json').write_text(json.dumps({'repo_id': restore_repo, 'path': restore_prefix, 'artifact_revision': restore_revision, 'receipt_revision': restore_receipt_revision, 'recovered_files_sha256': restore_hashes, 'prediction_hashes_verified': True, 'new_training_updates': 0, 'model_weights_downloaded': False}, indent=2) + '\n')
restore_archive = Path(str(restore_dir) + '-evidence.zip')
with zipfile.ZipFile(restore_archive, 'x', compression=zipfile.ZIP_DEFLATED) as restore_bundle:
    for restore_file in sorted(restore_dir.rglob('*')):
        if restore_file.is_file():
            restore_bundle.write(restore_file, arcname=str(restore_file.relative_to(restore_dir)))
restore_token = None
print('Recovered evidence only; no training or model-weight download.')
print('Reported full-M1 gate pass:', restore_summary['m1_candidate']['passed'])
print('Failed checks:', [key for key, value in restore_summary['m1_candidate']['checks'].items() if not value])
print('DOWNLOAD THIS ZIP:', restore_archive)


persistence.json:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

artifact_manifest.json:   0%|          | 0.00/830 [00:00<?, ?B/s]

Recovered 1/38: artifact_manifest.json


augmentation_audit.json:   0%|          | 0.00/2.39k [00:00<?, ?B/s]

Recovered 2/38: augmentation_audit.json


base_controls_answer_only.jsonl:   0%|          | 0.00/30.8k [00:00<?, ?B/s]

... (67 outputs left out)

summary.json:   0%|          | 0.00/46.3k [00:00<?, ?B/s]

Recovered 37/38: summary.json


training.jsonl:   0%|          | 0.00/347k [00:00<?, ?B/s]

Recovered 38/38: training.jsonl
Recovered evidence only; no training or model-weight download.
Reported full-M1 gate pass: False
Failed checks: ['per_task/thinking/F/prize_to_author']
DOWNLOAD THIS ZIP: /kaggle/working/m1-full-hpcajqfx-restored-lnl0x9a4-evidence.zip
